# Stable Baselines

Stable Baselines is a set of high-quality implementations of reinforcement learning algorithms in Python. It is built on top of OpenAI Gym (Gymnasium) and provides a user-friendly interface for training, evaluating, and deploying RL agents. Stable Baselines offers a wide range of popular RL algorithms, including Proximal Policy Optimization (PPO), Deep Q-Networks (DQN), and Trust Region Policy Optimization (TRPO), among others. It aims to provide reliable and stable implementations that are extensively tested and documented, making it easier for researchers and practitioners to work with reinforcement learning.


## Install Dependencies

In [ ]:
!pip install gymnasium
!pip install stable-baselines3

## Helpers



In [ ]:
# Video management imports
import cv2
import matplotlib.pyplot as plt

# Check if we running in Google Colab or Jupyter Notebook
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab')
    # Do you need to connect with Google Drive? Uncomment the next two lines
    # from google.colab import drive
    # drive.mount('/content/drive')
    # This auxiliary function simplifies the visualization of OpenCV Images
    from google.colab.patches import cv2_imshow
else:
    print('Running in Jupyter Notebook')
    # This auxiliary function simplifies the visualization of OpenCV Images
    def cv2_imshow(img, title=''):
        if img.ndim > 2:
            img= cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img)
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()
        else:
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img, cmap='gray')
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()

# Helper functions to save videos and images
def save_video(img_array, path='/content/test.mp4'):
  height, width, layers = img_array[0].shape
  size = (width, height)
  out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'MP4V'), 15, size)
  if out.isOpened():
    for i in range(len(img_array)):
      bgr_img = cv2.cvtColor(img_array[i], cv2.COLOR_RGB2BGR)
      out.write(bgr_img)
    out.release()
    print('Video saved.')
  else:
    print(f'Could not save video path: {path}')


# Training RL with Stable Baselines

First let's import the libraries. In this notebook, we include Stable Baselines

In [ ]:
# Check the version of the installed versions
import gymnasium as gym
import stable_baselines3 as sb3
import torch
import numpy as np

print(f"Gymnasium version: {gym.__version__}")
print(f"Stable Baselines version : {sb3.__version__}")
print(f"Torch version : {torch.__version__}")
print(f"Numpy version : {np.__version__}")

## Basic Training

To train an agent we only need an algorithm object an environment and a call to the learn function.

In [ ]:
from stable_baselines3 import A2C

env = gym.make("CartPole-v1", render_mode="rgb_array")

model = A2C("MlpPolicy", env, verbose=1, learning_rate=0.001)
model.learn(total_timesteps=10_000)

We can now test it. Notice how we use the predict function provided by the SB3 agent

In [ ]:
vec_env = model.get_env()
obs = vec_env.reset()
imgs = []
done = False
while not done:
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action) # Notice the difference with standard Gym environments
    img = vec_env.render("rgb_array")
    imgs.append(img)

env.close()

save_video(imgs, path='/content/sb3_a2c_cartpole.mp4')

## Custom network

We can create agents using a custom neural network. In the simplest cases we can define the network as a list of layer sizes:

In [ ]:
from stable_baselines3 import PPO

# Custom actor (pi) and value function (vf) networks
# of two layers of size 32 each with Relu activation function
# Note: an extra linear layer will be added on top of the pi and the vf nets, respectively
policy_kwargs = dict(activation_fn=torch.nn.ReLU,
                     net_arch=dict(pi=[64, 64], vf=[64, 64]))
# Create the agent
model = PPO("MlpPolicy", "CartPole-v1", policy_kwargs=policy_kwargs, verbose=1)
# Retrieve the environment
env = model.get_env()
# Train the agent
model.learn(total_timesteps=30_000)

Stable Baselines also enables to save and reload the model from a file

In [ ]:
# Save the agent
# model.save("ppo_cartpole")
# del model
# the policy_kwargs are automatically loaded
# model = PPO.load("ppo_cartpole", env=env)

Once trained, we can test it easily:

In [ ]:
vec_env = model.get_env()
obs = vec_env.reset()
imgs = []
done = False
while not done:
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action) # Notice the difference between
    img = vec_env.render("rgb_array")
    imgs.append(img)

env.close()

save_video(imgs, path='/content/sb3_ppo_cartpole.mp4')

Let's try another enviroment: Acrobot

In [ ]:
# Custom actor (pi) and value function (vf) networks
# of two layers of size 32 each with Relu activation function
# Note: an extra linear layer will be added on top of the pi and the vf nets, respectively
policy_kwargs = dict(activation_fn=torch.nn.ReLU,
                     net_arch=dict(pi=[64, 64], vf=[64, 64]))
# Create the agent
model = PPO("MlpPolicy", "Acrobot-v1", policy_kwargs=policy_kwargs, verbose=1)
# Retrieve the environment
env = model.get_env()
# Train the agent
model.learn(total_timesteps=30_000)

Let's see the results:

In [ ]:
vec_env = model.get_env()
obs = vec_env.reset()
imgs = []
done = False
while not done:
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action) # Notice the difference between
    img = vec_env.render("rgb_array")
    imgs.append(img)

env.close()

save_video(imgs, path='/content/sb3_ppo_acrobot.mp4')

## Using a custom network with Pytorch

We can train our agent with a custom network. In this case we will create a simple module for an actor-critic network. We need an adapter class *CustomActorCriticPolicy* to adapt our network to the needs of the algorithm and the environment.

In [ ]:
from typing import Callable, Dict, List, Optional, Tuple, Type, Union

from stable_baselines3.common.policies import ActorCriticPolicy
from gymnasium import spaces
import torch
from torch import nn

class CustomNetwork(nn.Module):
    """
    Custom network for policy and value function.
    It receives as input the features extracted by the features extractor.

    :param feature_dim: dimension of the features extracted with the features_extractor (e.g. features from a CNN)
    :param last_layer_dim_pi: (int) number of units for the last layer of the policy network
    :param last_layer_dim_vf: (int) number of units for the last layer of the value network
    """

    def __init__(
        self,
        obs_space_size: int,
        last_layer_dim_pi: int = 64,
        last_layer_dim_vf: int = 64,
    ):
        super().__init__()

        # IMPORTANT:
        # Save output dimensions, used to create the distributions
        self.latent_dim_pi = last_layer_dim_pi
        self.latent_dim_vf = last_layer_dim_vf

        # Policy network
        self.policy_net = nn.Sequential(
            nn.Linear(obs_space_size, last_layer_dim_pi), nn.ReLU()
        )
        # Value network
        self.value_net = nn.Sequential(
            nn.Linear(obs_space_size, last_layer_dim_vf), nn.ReLU()
        )

    def forward(self, features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        :return: (th.Tensor, th.Tensor) latent_policy, latent_value of the specified network.
            If all layers are shared, then ``latent_policy == latent_value``
        """
        return self.forward_actor(features), self.forward_critic(features)

    def forward_actor(self, features: torch.Tensor) -> torch.Tensor:
        return self.policy_net(features)

    def forward_critic(self, features: torch.Tensor) -> torch.Tensor:
        return self.value_net(features)

# This class will be automatically called by our algorithm with the adecuate
# dimensions
class CustomActorCriticPolicy(ActorCriticPolicy):
    def __init__(
        self,
        observation_space: spaces.Space,
        action_space: spaces.Space,
        lr_schedule: Callable[[float], float],
        *args,
        **kwargs,
    ):
        # Disable orthogonal initialization
        kwargs["ortho_init"] = False
        super().__init__(
            observation_space,
            action_space,
            lr_schedule,
            # Pass remaining arguments to base class
            *args,
            **kwargs,
        )


    def _build_mlp_extractor(self) -> None:
        self.mlp_extractor = CustomNetwork(self.features_dim)


model = PPO(CustomActorCriticPolicy, "CartPole-v1", verbose=0)
# Another way to show the progress of the training
model.learn(30_000, progress_bar=True)

In [ ]:
vec_env = model.get_env()
obs = vec_env.reset()
imgs = []
done = False
while not done:
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action) # Notice the difference between
    img = vec_env.render("rgb_array")
    imgs.append(img)

env.close()

save_video(imgs, path='/content/sb3_ppo_cartpole_cm.mp4')